# 🔤 Real Data Project — Bangla Spam Comment Detector
### NLP + TF-IDF + Logistic Regression on Real Data

---

**What you'll learn:**
- Why text needs special treatment before ML
- Text preprocessing — cleaning real data
- TF-IDF — converting text into numbers the model can read
- Building a full NLP pipeline (text → numbers → model → prediction)
- Evaluating on real imbalanced data
- Predicting on brand new comments

> 💡 This is your first **real-world ML project** — real data, real language, real problem.
> NLP skills are directly used in AI Engineering roles every day.

---
## 🧠 Section 1: Why Can't Models Read Text Directly?

ML models are pure math. They only understand numbers.

```
Model input:  [0.23, 1.45, 0.0, 3.2, ...]   ✅ numbers — model understands
Model input:  "আল্লাহ হেদায়েত দ্বান করুন"        ❌ text — model has no idea
```

So the job becomes: **convert text into a meaningful list of numbers.**

The most common technique for this is **TF-IDF**.

---

### What is TF-IDF?

TF-IDF stands for **Term Frequency — Inverse Document Frequency.**

Forget the name. Here's the idea with a simple example:

Imagine 3 comments:
```
Comment A: "ফলো করুন ফলো করুন ফলো করুন"   (follow follow follow — spam!)
Comment B: "খুব সুন্দর ভিডিও ভাই"            (very nice video bro — legit)
Comment C: "এই ভিডিও অনেক ভালো লাগলো"        (loved this video — legit)
```

**TF (Term Frequency):** How often does a word appear in THIS comment?
- "ফলো" appears 3 times in Comment A → high TF

**IDF (Inverse Document Frequency):** How rare is this word ACROSS ALL comments?
- "ভিডিও" appears in B and C → less unique, lower IDF
- "ফলো" appears only in A → very unique, higher IDF

**TF-IDF = TF × IDF**
Words that appear a lot IN a comment AND are rare ACROSS all comments get high scores.
Common words like "এই", "ভাই", "এবং" get low scores — they appear everywhere.

The result: each comment becomes a **vector of numbers** — one number per unique word.
Spam comments will have high scores for spam-like words.
The model learns which word patterns = spam.

---
## 📦 Section 2: Load and Explore the Real Dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ⚠️ UPDATE THIS PATH to where you saved the CSV
df = pd.read_csv("Bangla_Spam_Comment_Dataset_-_Original_.csv")

print(f"Shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print(f"\nClass distribution:")
print(df['Class'].value_counts())
print(f"\nMissing values:")
print(df.isnull().sum())
df.head(10)


In [ ]:
# ✅ Visualize class distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart
counts = df['Class'].value_counts()
axes[0].bar(['Not Spam (0)', 'Spam (1)'], counts.values, color=['#6366f1', '#ef4444'])
axes[0].set_title('Class Distribution')
axes[0].set_ylabel('Count')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 20, str(v), ha='center', fontweight='bold')

# Comment length distribution
df['comment_length'] = df['Comments'].astype(str).apply(len)
axes[1].hist(df[df['Class']==0]['comment_length'], bins=50,
             alpha=0.6, label='Not Spam', color='#6366f1')
axes[1].hist(df[df['Class']==1]['comment_length'], bins=50,
             alpha=0.6, label='Spam', color='#ef4444')
axes[1].set_title('Comment Length: Spam vs Not Spam')
axes[1].set_xlabel('Character Count')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"\nAvg length — Not Spam: {df[df['Class']==0]['comment_length'].mean():.0f} chars")
print(f"Avg length — Spam:     {df[df['Class']==1]['comment_length'].mean():.0f} chars")


In [ ]:
# ✅ Look at real examples — understand your data before modelling
# This is called EDA (Exploratory Data Analysis) — always do this first!

print("=== SPAM COMMENTS (Class = 1) ===")
for comment in df[df['Class']==1]['Comments'].head(5).tolist():
    print(f"  → {comment}")

print("\n=== NOT SPAM COMMENTS (Class = 0) ===")
for comment in df[df['Class']==0]['Comments'].head(5).tolist():
    print(f"  → {comment}")


---
## 🧹 Section 3: Text Preprocessing

Before converting text to numbers, we need to clean it.
Raw text has noise: extra spaces, punctuation, emojis, URLs.

Think of this like data cleaning in Pandas — but for text.

In [ ]:
import re

def clean_text(text):
    """
    Clean a comment before feeding it to the model.
    Works for Bangla + mixed text.
    """
    # Convert to string (in case of NaN)
    text = str(text)

    # Remove URLs
    text = re.sub(r'http\S+|www\S+', '', text)

    # Remove emojis and special symbols
    text = re.sub(r'[^\w\s\u0980-\u09FF]', ' ', text)
    # ↑ keeps: word chars, spaces, and the full Bangla unicode range

    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()

    return text


# Test it on a few examples
examples = [
    "ফেলো দিয়ে দিছি ভাই আপনিও দিয়ে দেন 🤟🤟🤟",
    "❗৫০ জিবি ❗ একদম ফ্রি!!! Visit: http://spam.com",
    "আল্লাহ আমাদের সবাই কে হেদায়াত দান করুক,আমিন...."
]

print("=== Cleaning Examples ===")
for ex in examples:
    cleaned = clean_text(ex)
    print(f"Before: {ex}")
    print(f"After:  {cleaned}")
    print()


In [ ]:
# ✅ Apply cleaning to the entire dataset
df['cleaned_comment'] = df['Comments'].apply(clean_text)

print("Cleaning done!")
print(f"Original:  {df['Comments'].iloc[6]}")
print(f"Cleaned:   {df['cleaned_comment'].iloc[6]}")


---
## 🔢 Section 4: TF-IDF Vectorization — Text to Numbers

Now the key step — converting cleaned text into numbers.

Think of TfidfVectorizer like `StandardScaler` but for text:
- `StandardScaler` transforms numbers to a standard scale
- `TfidfVectorizer` transforms text to a matrix of TF-IDF scores

Same rule applies:
- `fit_transform()` on training data only
- `transform()` on test data

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

# Features and Label
X = df['cleaned_comment']
y = df['Class']

# Train/Test split FIRST — before vectorizing!
# Why? To prevent data leakage — test text must not influence the vectorizer
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
    # stratify=y keeps the same spam ratio in train and test
)

print(f"Training comments: {len(X_train)}")
print(f"Testing comments:  {len(X_test)}")


In [ ]:
# ✅ Create and fit TF-IDF vectorizer

vectorizer = TfidfVectorizer(
    max_features=5000,    # keep only top 5000 most important words
    ngram_range=(1, 2),   # use single words AND two-word phrases
    min_df=2,             # ignore words appearing in less than 2 comments
)

# fit_transform on TRAIN — learn vocabulary from training data only
X_train_tfidf = vectorizer.fit_transform(X_train)

# transform on TEST — apply same vocabulary
X_test_tfidf = vectorizer.transform(X_test)

print(f"Training matrix shape: {X_train_tfidf.shape}")
# (7200, 5000) = 7200 comments, each represented by 5000 TF-IDF scores

print(f"\nSample vocabulary (first 20 words learned):")
print(list(vectorizer.vocabulary_.keys())[:20])


In [ ]:
# ✅ See what TF-IDF actually produces for one comment
# This makes the concept concrete

sample_comment = ["ফলো করুন ফলো করুন"]
sample_tfidf = vectorizer.transform(sample_comment)

# Get non-zero scores for this comment
feature_names = vectorizer.get_feature_names_out()
scores = sample_tfidf.toarray()[0]
nonzero = [(feature_names[i], round(scores[i], 4))
           for i in range(len(scores)) if scores[i] > 0]

print(f"Comment: '{sample_comment[0]}'")
print(f"\nTF-IDF scores (word → score):")
for word, score in sorted(nonzero, key=lambda x: -x[1]):
    print(f"  '{word}': {score}")
print(f"\nTotal features: {len(scores)}, Non-zero: {len(nonzero)}")
# Most scores are 0 — only words IN this comment get non-zero scores


---
## 🏋️ Section 5: Train the Spam Detector

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score,
    recall_score, f1_score, classification_report, confusion_matrix
)

# Same 3-step sklearn pattern — model doesn't care that input came from text!
model = LogisticRegression(random_state=42, max_iter=1000)
model.fit(X_train_tfidf, y_train)

# Predict
y_pred = model.predict(X_test_tfidf)

print("✅ Model trained on real Bangla comments!")
print(f"\nTest set size: {len(y_test)} comments")


In [ ]:
# ✅ Evaluate — these numbers will NOT be 100% like synthetic data!

print("=" * 50)
print("   REAL DATA SPAM DETECTOR — RESULTS")
print("=" * 50)
print(f"Accuracy:  {accuracy_score(y_test, y_pred)*100:.1f}%")
print(f"Precision: {precision_score(y_test, y_pred)*100:.1f}%")
print(f"Recall:    {recall_score(y_test, y_pred)*100:.1f}%")
print(f"F1 Score:  {f1_score(y_test, y_pred)*100:.1f}%")
print()
print(classification_report(y_test, y_pred, target_names=['Not Spam', 'Spam']))


In [ ]:
# ✅ Confusion Matrix

cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_xticklabels(['Predicted\nNot Spam', 'Predicted\nSpam'])
ax.set_yticklabels(['Actual\nNot Spam', 'Actual\nSpam'])
for i in range(2):
    for j in range(2):
        ax.text(j, i, str(cm[i, j]), ha='center', va='center',
                fontsize=20, fontweight='bold',
                color='white' if cm[i, j] > cm.max()/2 else 'black')
ax.set_title('Confusion Matrix — Bangla Spam Detector', fontweight='bold')
plt.colorbar(im)
plt.tight_layout()
plt.show()


In [ ]:
# ✅ See which words the model thinks are most spammy
# This is called 'model interpretability' — understanding WHY it predicts what it does

feature_names = vectorizer.get_feature_names_out()
coefficients = model.coef_[0]

# Top spam words (highest positive coefficients)
top_spam_idx = coefficients.argsort()[-15:][::-1]
top_spam_words = [(feature_names[i], round(coefficients[i], 3)) for i in top_spam_idx]

# Top not-spam words (highest negative coefficients)
top_legit_idx = coefficients.argsort()[:15]
top_legit_words = [(feature_names[i], round(coefficients[i], 3)) for i in top_legit_idx]

print("🚨 Top words that indicate SPAM:")
for word, score in top_spam_words:
    print(f"   '{word}': {score}")

print("\n✅ Top words that indicate NOT SPAM:")
for word, score in top_legit_words:
    print(f"   '{word}': {score}")


---
## 🎯 Section 6: Predict on Brand New Comments

In [ ]:
def predict_spam(comments):
    """
    Takes a list of raw comments,
    cleans them, vectorizes, predicts.
    This is what a production API would do!
    """
    cleaned = [clean_text(c) for c in comments]
    vectorized = vectorizer.transform(cleaned)
    predictions = model.predict(vectorized)
    probabilities = model.predict_proba(vectorized)

    print("=== SPAM DETECTION RESULTS ===")
    for comment, pred, prob in zip(comments, predictions, probabilities):
        label = "🚨 SPAM" if pred == 1 else "✅ NOT SPAM"
        short = comment[:60] + "..." if len(comment) > 60 else comment
        print(f"{label} ({prob[1]*100:.1f}% spam confidence)")
        print(f"   '{short}'")
        print()


# Test with new comments
new_comments = [
    "ফলো দিয়ে দেন ভাই আমিও ফলো দেবো",          # likely spam — follow exchange
    "এই ভিডিওটা অনেক ভালো লাগলো, ধন্যবাদ",       # likely not spam
    "❗ফ্রি ইন্টারনেট পেতে এখনই ক্লিক করুন❗",    # likely spam — free internet
    "আল্লাহ আপনাকে ভালো রাখুক ভাই",              # likely not spam
]

predict_spam(new_comments)


---
## 🎯 Section 7: Exercises

### Exercise 1
Change `max_features` in TfidfVectorizer from `5000` to `1000` and `10000`.
How does F1 score change? What does this tell you about vocabulary size?

### Exercise 2
Change `ngram_range=(1, 2)` to `ngram_range=(1, 1)` (single words only).
Then try `ngram_range=(1, 3)`. Does including 3-word phrases help?

### Exercise 3
Add a new feature: `comment_length` (number of characters) alongside TF-IDF.
```python
from scipy.sparse import hstack
import numpy as np

# Hint: combine TF-IDF matrix with length feature
train_lengths = np.array(X_train.apply(len)).reshape(-1, 1)
X_train_combined = hstack([X_train_tfidf, train_lengths])
```
Does adding comment length improve the F1 score?

> 💡 Exercise 3 teaches you **feature combination** — mixing text features with
> numeric features. This is a real production technique used in spam filters.

In [ ]:
# 🏋️ YOUR WORKSPACE

# ✏️ Exercise 1:


# ✏️ Exercise 2:


# ✏️ Exercise 3:



---
## ✅ Checklist

- [ ] Understand why text needs to be converted to numbers
- [ ] Can explain TF-IDF in plain words
- [ ] Know why train/test split happens BEFORE vectorizing
- [ ] Can read model coefficients to understand which words = spam
- [ ] Built a working Bangla spam detector on real data
- [ ] Completed all 3 exercises

---

## 🔜 What's Next

**Phase 2 — Week 5–6: Decision Trees + Random Forest**

A completely different model type that:
- Works on both numeric AND text features
- Is highly visual and interpretable
- Random Forest = many trees combined = much more powerful
- Project: Viral post predictor — your most powerful model yet

> 💬 Finish the exercises and come back!